# ML Regression Tutorial

This tutorial demonstrates how to use the HypEx ML module for regression tasks.
We build a pipeline that scales features, cross-validates several models,
selects the best one, and reports the results — all within the HypEx Executor framework.

<ul>
  <li><a href="#dataset-preparation">Dataset preparation</a></li>
  <li><a href="#building-the-ml-pipeline">Building the ML pipeline</a></li>
  <li><a href="#viewing-results-with-reporter">Viewing results with Reporter</a></li>
  <li><a href="#synthetic-regression">Synthetic regression</a></li>
</ul>

In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing, make_regression

from hypex.dataset import Dataset, ExperimentData, FeatureRole, TargetRole
from hypex.ml import (
    MSE,
    CatBoostPredictor,
    LinearRegressionPredictor,
    MLExperiment,
    ModelSelection,
    R2Score,
    RandomForestPredictor,
    RidgePredictor,
    StandardScaler,
)
from hypex.reporters import ModelSelectionReporter

## Dataset preparation

HypEx uses `Dataset` with role-based columns. Every column must have a role:
- **FeatureRole** — predictor variables used by models.
- **TargetRole** — the variable we want to predict.

Below we load the California Housing dataset and wrap it into a `Dataset`.

In [2]:
housing = fetch_california_housing(as_frame=True)

roles = {col: FeatureRole() for col in housing.feature_names}
roles["MedHouseVal"] = TargetRole()

data = Dataset(roles=roles, data=housing.frame)
data

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


## Building the ML experiment

`MLExperiment` chains `MLExecutor` steps sequentially. Each step receives
`ExperimentData` and passes it to the next one.

Our experiment has two steps:
1. **StandardScaler** — normalises numeric features (zero mean, unit variance).
2. **ModelSelection** — cross-validates several predictors, picks the best by the
   chosen metric, fits it on the full training set, and computes final metrics.

`ModelSelection` accepts:
- `predictors` — a list of `SklearnPredictor` instances to compare.
- `metrics` — metric executors (`R2Score`, `MSE`, `MAE`, `RMSE`).
- `cv` — number of cross-validation folds.
- `main_metric` — the metric used to select the best model.

In [4]:
experiment = MLExperiment(steps=[
    StandardScaler(),
    ModelSelection(
        predictors=[
            LinearRegressionPredictor(),
            RidgePredictor(alpha=1.0),
            RandomForestPredictor(n_estimators=50),
            CatBoostPredictor(iterations=100),
        ],
        metrics=[R2Score(), MSE()],
        cv=5,
        main_metric="r2",
    ),
])

result = experiment.execute(ExperimentData(data))

## Viewing results with Reporter

Following the HypEx architecture, result formatting is handled by **Reporters**,
not by the executors themselves. `ModelSelectionReporter` reads raw CV results
from `ExperimentData.variables` and builds a readable table with columns:
- `model` — predictor name.
- `<metric>_mean` / `<metric>_std` — cross-validation mean and standard deviation.
- `best` — marks the model selected as best.

In [5]:
reporter = ModelSelectionReporter()
cv_table = reporter.report(result)
cv_table

,model,r2_mean,r2_std,mse_mean,mse_std,best
0,LinearRegressionPredictor_0,0.553031,0.061692,0.558290,0.065602,False
1,RidgePredictor_1,0.553038,0.061703,0.558282,0.065624,False
2,RandomForestPredictor_2,0.649863,0.074306,0.433607,0.059301,False
3,CatBoostPredictor_3,0.658333,0.076069,0.424115,0.070350,True


The best predictor and final (full-data) metrics are also available
programmatically from the pipeline objects.

In [6]:
selection = experiment.steps[1]
print(f"Best predictor: {selection.best_predictor_.__class__.__name__}")
print(f"Final R2:  {result.variables[selection.metric_executors[0].id]['r2']:.4f}")
print(f"Final MSE: {result.variables[selection.metric_executors[1].id]['mse']:.4f}")

Best predictor: CatBoostPredictor
Final R2:  0.8842
Final MSE: 0.1542


## Synthetic regression

Below we repeat the experiment on a synthetic dataset generated with
`sklearn.datasets.make_regression`. This time we select the best model
by **MSE** (lower is better) and compare five predictors including two
Ridge variants with different regularisation strengths and a CatBoost model.

In [7]:
features, target = make_regression(
    n_samples=1000, n_features=10, n_informative=5,
    noise=10.0, random_state=42,
)
feature_cols = [f"f{i}" for i in range(features.shape[1])]
df = pd.DataFrame(features, columns=feature_cols)
df["target"] = target

roles = {col: FeatureRole() for col in feature_cols}
roles["target"] = TargetRole()

synth_data = Dataset(roles=roles, data=df)
synth_data

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,target
0,0.123429,-1.253402,0.370340,0.101788,0.092628,-0.589254,0.306348,-1.458213,1.630130,1.242863,18.926845
1,-0.216827,0.214983,-0.028817,-1.701140,0.264482,0.314972,0.374062,-0.292758,0.501900,0.063702,-56.463272
2,0.388979,1.655407,-0.755792,-1.161784,-0.300860,1.048707,-0.283139,0.251474,-0.194269,-1.209477,-28.971405
3,0.219072,-0.251552,-1.095871,-0.806520,-0.435139,2.768374,1.677201,-0.475392,-0.835870,-1.314879,-0.761331
4,0.715734,0.383168,-0.686715,-1.236136,0.731001,1.623885,1.254338,2.394362,2.185095,0.538435,6.578192
...,...,...,...,...,...,...,...,...,...,...,...
995,0.332314,0.067518,0.115675,0.711615,-1.534114,1.179297,-1.124642,-0.748487,1.551152,1.277677,41.652693
996,-0.214150,-0.837090,-1.585626,-1.114081,-0.942060,1.140068,-0.630931,0.837154,-0.321159,-0.547996,-69.267643
997,1.368079,-0.592241,1.223856,-0.812546,0.034888,0.337956,2.105297,0.125901,0.851124,-1.375511,65.937223
998,1.223083,0.270057,-0.570351,-0.824514,-0.528978,-1.820603,-0.608590,-0.258854,0.352505,-1.056558,14.042409


In [8]:
experiment2 = MLExperiment(steps=[
    StandardScaler(),
    ModelSelection(
        predictors=[
            LinearRegressionPredictor(),
            RidgePredictor(alpha=0.5),
            RidgePredictor(alpha=10.0),
            RandomForestPredictor(n_estimators=100),
            CatBoostPredictor(iterations=200, learning_rate=0.05),
        ],
        metrics=[R2Score(), MSE()],
        cv=5,
        main_metric="mse",
    ),
])

result2 = experiment2.execute(ExperimentData(synth_data))

In [9]:
reporter.report(result2)

,model,r2_mean,r2_std,mse_mean,mse_std,best
0,LinearRegressionPredictor_0,0.979154,0.003829,99.005462,9.977098,True
1,RidgePredictor_1,0.979153,0.003836,99.011574,10.005363,False
2,RidgePredictor_2,0.978969,0.003953,99.859550,10.542916,False
3,RandomForestPredictor_3,0.936171,0.007227,311.599896,71.832040,False
4,CatBoostPredictor_4,0.961575,0.011682,192.963151,90.154708,False


In [10]:
selection2 = experiment2.steps[1]
print(f"Best predictor: {selection2.best_predictor_.__class__.__name__}")

Best predictor: LinearRegressionPredictor


## Summary

The HypEx ML module provides:

- **MLExperiment** — sequential composition of ML steps (the ML experiment orchestrator).
- **StandardScaler** — feature normalisation (`MLTransformer`).
- **ModelSelection** — cross-validation, model selection, and final evaluation.
- **ModelSelectionReporter** — formatted result tables following the Reporter pattern.
- **Metric executors** (`R2Score`, `MSE`, `MAE`, `RMSE`) — pluggable metrics.
- **Predictors** (`LinearRegressionPredictor`, `RidgePredictor`,
  `RandomForestPredictor`, `CatBoostPredictor`) — model wrappers via Extensions.

All sklearn-specific code is isolated in Extensions, keeping Executors
backend-agnostic and following the HypEx architecture.